# 2교시 · API 문서 분석과 요청 구문 설계

**이 시간에 할 일**

1. API 문서를 언어모델에게 읽혀 **요청 명세서**로 정리시킵니다
2. 정리된 명세서를 **실제 호출 한 건으로 검증**합니다
3. 연구 주제에서 API 파라미터까지 내려오는 변환표를 만듭니다

---

**여기서 언어모델을 처음 씁니다. 대화 상대가 아니라 문서 파서로 씁니다.**

API 키가 없어도 전부 따라오실 수 있게 두 갈래로 만들어 뒀습니다.

- **경로 A (키 없음)** — 노트북이 프롬프트를 만들어 출력합니다. 복사해서 쓰시던 챗 화면에 붙여넣고, 결과를 다시 노트북에 붙여넣습니다
- **경로 B (키 있음)** — 노트북이 직접 호출합니다

In [ ]:
import json
import re
import textwrap
import pandas as pd
import requests

ANTHROPIC_API_KEY = ""   # 비워 두면 경로 A로 진행됩니다
MODEL = "claude-sonnet-4-6"

print("경로 B (자동)" if ANTHROPIC_API_KEY else "경로 A (프롬프트 복사)")

## 2-1. 명세서에 무엇이 들어가야 하는가

먼저 우리가 **무엇을 뽑아낼 것인지** 정해야 합니다. 이걸 정하지 않고 "이 문서 정리해줘"라고 하면 매번 다른 모양이 나옵니다.

수집 코드를 짜는 데 반드시 필요한 항목은 여덟 개입니다.

In [ ]:
SPEC_SCHEMA = {
    "source_name":     "기관명",
    "endpoint":        "요청 주소 (경로 변수는 {중괄호}로 표시)",
    "auth":            {"required": "true/false", "param_name": "인증키 파라미터명", "how": "쿼리스트링/헤더"},
    "params_required": [{"name": "", "meaning": "", "example": ""}],
    "params_optional": [{"name": "", "meaning": "", "example": ""}],
    "response_shape":  "최상위 구조 설명 (리스트인지 딕셔너리인지, 몇 겹인지)",
    "data_path":       "관측치 목록에 닿는 경로 (예: [1] 또는 ['data']['items'])",
    "error_codes":     [{"code": "", "meaning": ""}],
    "pagination":      "페이지 처리 방식과 총건수 확인 방법",
    "rate_limit":      "호출 제한",
}

print(json.dumps(SPEC_SCHEMA, indent=2, ensure_ascii=False))

### 왜 이 여덟 개인가

`data_path`와 `pagination`이 특히 중요합니다.

1교시에서 World Bank 응답 최상위가 리스트 2원소였던 걸 보셨습니다. `data_path`가 없으면 그 함정에 매번 다시 빠집니다. `pagination`이 없으면 앞장만 받고 끝납니다.

**명세서 양식을 고정하는 것 자체가 실패 방지 장치입니다.**

## 2-2. 프롬프트 만들기

문서 텍스트를 넣으면 프롬프트를 조립해 주는 함수입니다.

In [ ]:
def build_spec_prompt(doc_text, source_name):
    return textwrap.dedent(f"""\
    아래는 {source_name}의 API 문서입니다. 이 문서를 읽고 요청 명세서를 JSON으로 정리하세요.

    규칙:
    - 아래 스키마의 키를 그대로 쓰고, 키를 추가하거나 빼지 마세요.
    - 문서에 없는 항목은 값을 "문서에 없음"으로 채우세요. 추측해서 채우지 마세요.
    - 설명이나 머리말 없이 JSON만 출력하세요. 코드펜스도 붙이지 마세요.

    스키마:
    {json.dumps(SPEC_SCHEMA, indent=2, ensure_ascii=False)}

    문서:
    ---
    {doc_text}
    ---
    """)


# 실습용 문서 발췌 (World Bank Indicators API)
WB_DOC = """
World Bank Indicators API

Base URL: https://api.worldbank.org/v2/country/{country}/indicator/{indicator}

country: ISO 3166-1 alpha-3 code. Multiple countries can be joined with a semicolon.
         Use "all" for every country.
indicator: World Bank indicator code, e.g. SP.DYN.TFRT.IN

Query parameters:
  format    json or xml. Default is xml.
  date      Year or range, e.g. 2015 or 2015:2025
  per_page  Number of records per page. Default 50.
  page      Page number, starting at 1.

The JSON response is an array of two elements. The first element is an object with
paging information: page, pages, per_page, total, lastupdated. The second element is
an array of observation objects, each containing indicator, country, countryiso3code,
date, value, unit, obs_status and decimal. A missing observation has value null.

No API key is required. Requests are not authenticated.
"""

prompt = build_spec_prompt(WB_DOC, "World Bank")
print(prompt[:600], "\n...(생략)")

### 프롬프트에서 눈여겨볼 세 줄

- **"키를 그대로 쓰고, 추가하거나 빼지 마세요"** — 출력 모양을 고정합니다
- **"문서에 없는 항목은 '문서에 없음'으로. 추측하지 마세요"** — 없는 걸 지어내는 걸 막습니다. 이게 없으면 그럴듯한 파라미터를 만들어 냅니다
- **"JSON만 출력. 코드펜스도 붙이지 마세요"** — 파싱을 위한 요구입니다

세 줄 다 **나중에 코드가 이 결과를 먹기 때문에** 필요한 것입니다.

## 2-3. 실행

경로 A는 아래 출력을 복사해서 쓰시던 챗 화면에 붙여넣으세요.

In [ ]:
def ask_llm(prompt_text):
    """경로 B. 키가 있으면 직접 호출합니다."""
    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={"model": MODEL, "max_tokens": 2000,
              "messages": [{"role": "user", "content": prompt_text}]},
        timeout=60,
    )
    r.raise_for_status()
    return "".join(b["text"] for b in r.json()["content"] if b["type"] == "text")


def parse_spec(raw_text):
    """코드펜스가 붙어 와도 JSON만 건져냅니다."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw_text.strip(), flags=re.M).strip()
    return json.loads(cleaned)


if ANTHROPIC_API_KEY:
    spec = parse_spec(ask_llm(prompt))
    print(json.dumps(spec, indent=2, ensure_ascii=False))
else:
    print("아래 프롬프트를 복사해서 붙여넣으세요.")
    print("=" * 70)
    print(prompt)
    print("=" * 70)

In [ ]:
# 경로 A: 받은 JSON을 아래 따옴표 사이에 붙여넣고 실행하세요.
PASTED = r"""
"""

if not ANTHROPIC_API_KEY:
    if PASTED.strip():
        spec = parse_spec(PASTED)
    else:
        # 붙여넣기 전에도 다음 셀이 돌아가도록 미리 채워둔 예시입니다
        spec = {
            "source_name": "World Bank",
            "endpoint": "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}",
            "auth": {"required": "false", "param_name": "문서에 없음", "how": "문서에 없음"},
            "params_required": [
                {"name": "country", "meaning": "ISO3 국가코드, 세미콜론으로 복수 지정", "example": "KOR;JPN"},
                {"name": "indicator", "meaning": "World Bank 지표 코드", "example": "SP.DYN.TFRT.IN"},
            ],
            "params_optional": [
                {"name": "format", "meaning": "응답 형식. 기본값은 xml", "example": "json"},
                {"name": "date", "meaning": "연도 또는 범위", "example": "2015:2025"},
                {"name": "per_page", "meaning": "쪽당 건수. 기본 50", "example": "500"},
                {"name": "page", "meaning": "쪽 번호. 1부터", "example": "1"},
            ],
            "response_shape": "원소 2개짜리 배열. 0번은 페이징 정보 객체, 1번은 관측치 배열",
            "data_path": "[1]",
            "error_codes": [{"code": "문서에 없음", "meaning": "문서에 없음"}],
            "pagination": "page/per_page. 응답 [0]의 total과 pages로 남은 쪽 확인",
            "rate_limit": "문서에 없음",
        }
        print("붙여넣기 전이라 예시 명세서로 진행합니다.\n")

print(json.dumps(spec, indent=2, ensure_ascii=False))

## 2-4. 명세서를 실제 호출로 검증합니다

**이 시간에서 가장 중요한 셀입니다.**

언어모델이 정리한 명세서는 아직 주장일 뿐입니다. 문서가 오래됐거나, 모델이 잘못 읽었거나, API가 그 사이에 바뀌었을 수 있습니다.

**문서를 읽혔으면 반드시 한 번 때려보고 넘어갑니다.** 문서와 응답이 다르면 응답이 맞습니다.

In [ ]:
def resolve_path(payload, path_expr):
    """명세서의 data_path 표기를 따라 관측치 목록까지 내려갑니다."""
    node = payload
    for key in re.findall(r"\[([^\]]+)\]", path_expr):
        key = key.strip("'\"")
        node = node[int(key)] if key.lstrip("-").isdigit() else node[key]
    return node


def verify_spec(spec, sample_values, expect_keys):
    url = spec["endpoint"].format(**sample_values["path"])
    report = {"url": url, "checks": []}

    def check(name, ok, detail=""):
        report["checks"].append({"항목": name, "결과": "통과" if ok else "불일치", "비고": detail})

    try:
        r = requests.get(url, params=sample_values["query"], timeout=20)
    except Exception as e:
        check("호출 성공", False, f"{type(e).__name__}: {e}")
        return report

    check("HTTP 200", r.status_code == 200, f"status={r.status_code}")
    if r.status_code != 200:
        return report

    try:
        payload = r.json()
    except Exception:
        check("JSON 파싱", False, "본문이 JSON이 아닙니다")
        return report
    check("JSON 파싱", True)

    shape_ok = isinstance(payload, list) and len(payload) == 2
    check("응답 최상위 구조", shape_ok, f"실제 타입 {type(payload).__name__}, 길이 {len(payload) if isinstance(payload, list) else '-'}")

    try:
        rows = resolve_path(payload, spec["data_path"])
        check("data_path 유효", isinstance(rows, list) and len(rows) > 0, f"{len(rows)}건")
    except Exception as e:
        check("data_path 유효", False, str(e))
        return report

    missing = [k for k in expect_keys if k not in rows[0]]
    check("관측치 필수 키", not missing, f"누락 {missing}" if missing else "전부 존재")

    meta = payload[0] if shape_ok else {}
    total, per_page = meta.get("total"), meta.get("per_page")
    if total is not None and per_page is not None:
        check("페이징 여유", int(total) <= int(per_page),
              f"total={total}, per_page={per_page}" + ("" if int(total) <= int(per_page) else " → 뒷장 남음"))
    return report


result = verify_spec(
    spec,
    sample_values={
        "path": {"country": "KOR", "indicator": "SP.DYN.TFRT.IN"},
        "query": {"format": "json", "date": "2015:2025", "per_page": 500},
    },
    expect_keys=["countryiso3code", "date", "value", "indicator"],
)

print("요청:", result["url"], "\n")
pd.DataFrame(result["checks"])

### 검증 결과를 명세서에 되먹입니다

불일치가 나오면 **명세서를 고칩니다.** 코드를 고치는 게 아닙니다.

명세서가 나중에 파이프라인의 `references/api-registry.md`가 되기 때문입니다. 여기서 정확하게 고쳐두면 4교시에 그대로 쓰입니다.

In [ ]:
spec["verified_at"] = pd.Timestamp.now(tz="Asia/Seoul").isoformat()
spec["verified_result"] = {r["항목"]: r["결과"] for r in result["checks"]}

with open("spec_worldbank.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, indent=2, ensure_ascii=False)

print("spec_worldbank.json 저장됨")
print("확인일시:", spec["verified_at"])

## 2-5. 연구 주제에서 파라미터까지

이제 방향을 뒤집습니다. 문서에서 시작하는 게 아니라 **연구 질문에서 시작해서** 파라미터까지 내려옵니다.

실무에서 실제로 하는 순서고, 여기서 빠뜨린 결정이 나중에 데이터를 다시 받게 만듭니다.

In [ ]:
worksheet = pd.DataFrame([
    {"결정 항목": "연구 질문",   "우리 값": "한국의 저출생은 다른 나라와 얼마나 다른가",
     "파라미터로": "-",              "빠뜨리면": "-"},
    {"결정 항목": "필요 지표",   "우리 값": "합계출산율",
     "파라미터로": "indicator",      "빠뜨리면": "-"},
    {"결정 항목": "비교 대상",   "우리 값": "OECD 주요 7개국 + 한국",
     "파라미터로": "country",        "빠뜨리면": "나중에 국가 추가하며 재수집"},
    {"결정 항목": "분석 기간",   "우리 값": "2015~2025",
     "파라미터로": "date",           "빠뜨리면": "추세 구간이 짧아 재수집"},
    {"결정 항목": "주기",        "우리 값": "연간",
     "파라미터로": "-",              "빠뜨리면": "분기 자료와 붙일 때 집계 기준 충돌"},
    {"결정 항목": "지역 단위",   "우리 값": "국가 (국내는 시도)",
     "파라미터로": "country / objL1", "빠뜨리면": "국내외 결합 시 키가 안 맞음"},
    {"결정 항목": "단위",        "우리 값": "명",
     "파라미터로": "-",              "빠뜨리면": "환산 누락으로 값이 틀림"},
    {"결정 항목": "기준",        "우리 값": "확정치 우선, 없으면 잠정치 표시",
     "파라미터로": "-",              "빠뜨리면": "재현이 안 됨"},
])
worksheet

**"빠뜨리면" 열을 보세요.** 전부 데이터를 다시 받아야 하는 사유입니다.

수집 전에 이 표를 채우는 데 10분이 걸립니다. 안 채우면 나중에 반나절을 다시 씁니다.

---

## 정리

- 명세서 **양식을 먼저 고정**하고 문서를 읽힙니다. 양식이 없으면 매번 다른 게 나옵니다
- "문서에 없으면 없다고 하라"를 프롬프트에 넣습니다. 없으면 지어냅니다
- **정리된 명세서는 주장입니다. 호출 한 건으로 검증한 뒤에 씁니다**
- 불일치가 나오면 명세서를 고칩니다. 이 명세서가 4교시 파이프라인의 소스 사전이 됩니다
- 수집 전에 연구 질문에서 파라미터까지 내려오는 표를 채웁니다

점심 이후에는 이 명세서를 재료로 수집 코드를 만듭니다.